# Data Linkage and cleaning

---

First I have standardised the Account_id outputs with 'ACC-XXXXXXXX'

Then, I have removed all exactly identical rows - not rows that just share the same account number in order to avoid removing frequent loanees

Then I have combined all the columns into a large table, which combines all the values from all 3 tables

# Loading the datasets

In [3]:
from pathlib import Path
import pandas as pd


def find_data_directory(start: Path) -> Path:
    """Find the project data folder from the current notebook location."""
    required_files = {
        "applications.csv",
        "performance_12m.csv",
        "experian_consumer.csv",
    }

    for candidate in [start, *start.parents]:
        data_directory = candidate / "data"

        if data_directory.is_dir():
            available_files = {
                path.name for path in data_directory.iterdir()
            }

            if required_files.issubset(available_files):
                return data_directory

    raise FileNotFoundError(
        "Could not find a data folder containing "
        "applications.csv, performance_12m.csv and "
        "experian_consumer.csv."
    )


DATA_DIR = find_data_directory(Path.cwd().resolve())

applications = pd.read_csv(
    DATA_DIR / "applications.csv",
    dtype={"account_id": "string"},
)

performance = pd.read_csv(
    DATA_DIR / "performance_12m.csv",
    dtype={"account_id": "string"},
)

experian_consumer = pd.read_csv(
    DATA_DIR / "experian_consumer.csv",
    dtype={"account_id": "string"},
)


print(f"Data loaded from: {DATA_DIR}")
print(f"Applications: {applications.shape}")
print(f"Performance:  {performance.shape}")
print(f"Experian:     {experian_consumer.shape}")

Data loaded from: C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data
Applications: (9987, 38)
Performance:  (10012, 13)
Experian:     (10000, 15)


---

# Cleaning and joining the data

In [5]:
from pathlib import Path
import pandas as pd


# Find the project root whether the notebook is opened from the root
# or from the notebooks folder.
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "pyproject.toml").is_file()
        ):
            return candidate

    raise FileNotFoundError("Could not find the project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data"

ACCOUNT_ID_PATTERN = r"^ACC-\d{8}$"


# Load account IDs as text so their formatting is preserved.
applications = pd.read_csv(
    DATA_DIR / "applications.csv",
    dtype={"account_id": "string"},
)

performance = pd.read_csv(
    DATA_DIR / "performance_12m.csv",
    dtype={"account_id": "string"},
)

delphi_raw = pd.read_csv(
    DATA_DIR / "experian_consumer.csv",
    dtype={"account_id": "string"},
)


def make_exception_rows(
    dataframe,
    source_table,
    reason,
):
    """Copy excluded rows and record why they were removed."""

    exceptions = dataframe.loc[reason.notna()].copy()

    # Add 2 because pandas starts at zero and CSV row 1 is the header.
    exceptions.insert(
        0,
        "source_row_number",
        exceptions.index + 2,
    )

    exceptions.insert(
        0,
        "exception_reason",
        reason.loc[exceptions.index].astype(str),
    )

    exceptions.insert(
        0,
        "source_table",
        source_table,
    )

    return exceptions


def clean_source(
    dataframe,
    source_table,
    extra_invalid=None,
    extra_reason=None,
):
    """
    Remove rows containing:
    - Missing account IDs
    - Invalid account IDs
    - Any duplicated account ID

    duplicated(keep=False) removes every copy of a duplicated ID.
    """

    account_id = dataframe["account_id"]

    missing_id = (
        account_id.isna()
        | account_id.str.strip().eq("")
    )

    invalid_id = ~account_id.str.fullmatch(
        ACCOUNT_ID_PATTERN,
        na=False,
    )

    duplicate_id = (
        account_id.duplicated(keep=False)
        & ~missing_id
        & ~invalid_id
    )

    reason = pd.Series(
        pd.NA,
        index=dataframe.index,
        dtype="string",
    )

    reason.loc[missing_id] = "missing account_id"

    reason.loc[~missing_id & invalid_id] = (
        "invalid account_id format"
    )

    reason.loc[
        ~missing_id & ~invalid_id & duplicate_id
    ] = "duplicate account_id"

    # This allows source-specific validation, such as invalid
    # performance values or missing Delphi scores.
    if extra_invalid is not None:
        mask = reason.isna() & extra_invalid
        reason.loc[mask] = extra_reason

    exceptions = make_exception_rows(
        dataframe,
        source_table,
        reason,
    )

    clean_data = dataframe.loc[reason.isna()].copy()

    return clean_data, exceptions

## Data Linkage for the working set

This checks performance values, retains only the Delphi score, identifies unmatched records and performs exact matching.

## Data Cleaning

- Remove duplicates
- Trim and Case match
- Restore canonical format
- Remove the excess rows and columns in experian data
- Remove all applications without performance data

In [11]:
from pathlib import Path
import pandas as pd


def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (
            (candidate / "data").is_dir()
            and (candidate / "pyproject.toml").is_file()
        ):
            return candidate

    raise FileNotFoundError("Could not find the project root")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data"

# Recognises ACC-12345678, ACC - 12345678 and acc 12345678.
INPUT_ID_PATTERN = r"ACC(?:\s*-\s*|\s+)([0-9]{8})"
CANONICAL_ID_PATTERN = r"ACC-[0-9]{8}"


applications = pd.read_csv(
    DATA_DIR / "applications.csv",
    dtype={"account_id": "string"},
)

performance = pd.read_csv(
    DATA_DIR / "performance_12m.csv",
    dtype={"account_id": "string"},
)

delphi_raw = pd.read_csv(
    DATA_DIR / "experian_consumer.csv",
    dtype={"account_id": "string"},
)


def normalise_account_ids(account_ids):
    """Repair recognised formatting only; never fuzzy-match."""

    text = account_ids.astype("string").str.strip().str.upper()

    digits = text.str.extract(
        rf"^{INPUT_ID_PATTERN}$",
        expand=False,
    )

    # Unrecognised formats become missing canonical IDs.
    return ("ACC-" + digits).astype("string")


def make_exception_rows(dataframe, source_table, reason):
    """Preserve excluded rows and their removal reasons."""

    excluded = dataframe.loc[reason.notna()].copy()

    excluded.insert(
        0,
        "source_row_number",
        excluded.index + 2,
    )

    excluded.insert(
        0,
        "exception_reason",
        reason.loc[excluded.index],
    )

    excluded.insert(0, "source_table", source_table)

    return excluded


def clean_source(
    dataframe,
    source_table,
    extra_invalid=None,
    extra_reason=None,
):
    dataframe = dataframe.copy()

    # Preserve the submitted ID for auditing.
    dataframe["original_account_id"] = dataframe["account_id"]

    raw_ids = dataframe["original_account_id"]
    canonical_ids = normalise_account_ids(raw_ids)

    missing_id = (
        raw_ids.isna()
        | raw_ids.str.strip().eq("").fillna(False)
    )

    invalid_id = ~missing_id & canonical_ids.isna()

    # Check duplicates AFTER normalisation.
    # keep=False rejects every occurrence, not just later copies.
    duplicate_id = (
        canonical_ids.notna()
        & canonical_ids.duplicated(keep=False)
    )

    dataframe["account_id"] = canonical_ids

    reasons = pd.Series(
        pd.NA,
        index=dataframe.index,
        dtype="string",
    )

    reasons.loc[missing_id] = "missing account_id"
    reasons.loc[invalid_id] = "invalid account_id format"
    reasons.loc[duplicate_id] = "duplicate account_id after normalisation"

    if extra_invalid is not None:
        mask = reasons.isna() & extra_invalid
        reasons.loc[mask] = extra_reason

    exceptions = make_exception_rows(
        dataframe,
        source_table,
        reasons,
    )

    clean_data = dataframe.loc[reasons.isna()].copy()

    return clean_data, exceptions


# Check that the required source columns exist.
for source_name, dataframe in {
    "applications": applications,
    "performance_12m": performance,
    "experian_consumer": delphi_raw,
}.items():
    if "account_id" not in dataframe.columns:
        raise ValueError(f"{source_name} is missing account_id")


# Require all twelve monthly performance fields.
performance_columns = [
    f"missed_payment_m{month:02d}"
    for month in range(1, 13)
]

missing_columns = [
    column
    for column in performance_columns
    if column not in performance.columns
]

if missing_columns:
    raise ValueError(
        f"Missing performance columns: {missing_columns}"
    )


# Invalid or missing monthly values are excluded.
invalid_performance = ~performance[
    performance_columns
].isin([0, 1]).all(axis=1)


# Ignore every detailed Delphi field.
if "DelphiScore" not in delphi_raw.columns:
    raise ValueError("Delphi source is missing DelphiScore")

delphi = delphi_raw[["account_id", "DelphiScore"]].copy()

numeric_delphi_score = pd.to_numeric(
    delphi["DelphiScore"],
    errors="coerce",
)

invalid_delphi = numeric_delphi_score.isna()


applications_clean, application_exceptions = clean_source(
    applications,
    "applications",
)

performance_clean, performance_exceptions = clean_source(
    performance,
    "performance_12m",
    extra_invalid=invalid_performance,
    extra_reason="invalid or missing monthly performance value",
)

delphi_clean, delphi_exceptions = clean_source(
    delphi,
    "experian_consumer",
    extra_invalid=invalid_delphi,
    extra_reason="missing or non-numeric DelphiScore",
)

delphi_clean["DelphiScore"] = pd.to_numeric(
    delphi_clean["DelphiScore"]
)

## Linkage and tabling

- Keep only rows with valid, unique and exact match across all 3 clean sources
- Move unmatched rows to 'Exceptions'

In [12]:
clean_tables = {
    "applications": applications_clean,
    "performance_12m": performance_clean,
    "experian_consumer": delphi_clean,
}

clean_id_sets = {
    source: set(dataframe["account_id"])
    for source, dataframe in clean_tables.items()
}


def unmatched_reason(account_ids, other_sources):
    reasons = []

    for account_id in account_ids:
        missing_from = [
            source
            for source, ids in other_sources.items()
            if account_id not in ids
        ]

        reasons.append(
            "no exact match in " + " and ".join(missing_from)
            if missing_from
            else pd.NA
        )

    return pd.Series(
        reasons,
        index=account_ids.index,
        dtype="string",
    )


exception_parts = [
    application_exceptions,
    performance_exceptions,
    delphi_exceptions,
]

# Record every valid-looking row that cannot join all sources.
for source, dataframe in clean_tables.items():
    other_sources = {
        other_source: ids
        for other_source, ids in clean_id_sets.items()
        if other_source != source
    }

    reasons = unmatched_reason(
        dataframe["account_id"],
        other_sources,
    )

    exception_parts.append(
        make_exception_rows(
            dataframe,
            source,
            reasons,
        )
    )


common_ids = set.intersection(*clean_id_sets.values())


# Retain canonical IDs in working data.
# Original IDs remain available in the exceptions audit.
tables_for_join = {
    source: dataframe.loc[
        dataframe["account_id"].isin(common_ids)
    ].drop(columns="original_account_id")
    for source, dataframe in clean_tables.items()
}


working_data = (
    tables_for_join["applications"]
    .merge(
        tables_for_join["performance_12m"],
        on="account_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        tables_for_join["experian_consumer"],
        on="account_id",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("account_id")
    .reset_index(drop=True)
)


exceptions = (
    pd.concat(
        exception_parts,
        ignore_index=True,
        sort=False,
    )
    .sort_values(
        ["source_table", "source_row_number"],
        kind="stable",
    )
    .reset_index(drop=True)
)

## Dataset generation

- Validate the results
- Push new tables into .csv's

In [13]:
from IPython.display import display


# Working data must contain unique, valid canonical IDs.
assert working_data["account_id"].is_unique
assert working_data["account_id"].notna().all()

assert working_data["account_id"].str.fullmatch(
    CANONICAL_ID_PATTERN
).all()

assert len(working_data) == len(common_ids)

assert working_data[performance_columns].isin(
    [0, 1]
).all().all()

assert working_data["DelphiScore"].notna().all()


# Ensure ignored Delphi columns did not enter working data.
ignored_delphi_columns = (
    set(delphi_raw.columns)
    - {"account_id", "DelphiScore"}
)

assert not (
    ignored_delphi_columns
    & set(working_data.columns)
)


# Every excluded source row must have a reason.
assert exceptions["exception_reason"].notna().all()

# A source row must appear in exceptions at most once.
assert not exceptions.duplicated(
    subset=["source_table", "source_row_number"]
).any()


# Every working account consumes one row from each source.
# All remaining source rows must appear in exceptions.
source_counts = {
    "applications": len(applications),
    "performance_12m": len(performance),
    "experian_consumer": len(delphi_raw),
}

for source, source_count in source_counts.items():
    excluded_count = exceptions["source_table"].eq(source).sum()

    assert source_count == len(working_data) + excluded_count, (
        f"Row reconciliation failed for {source}"
    )


# Create the new CSV files without changing the source CSVs.
working_path = DATA_DIR / "working data.csv"
exceptions_path = DATA_DIR / "exceptions.csv"

working_data.to_csv(working_path, index=False)
exceptions.to_csv(exceptions_path, index=False)


print(f"Working accounts: {len(working_data):,}")
print(f"Excluded source rows: {len(exceptions):,}")
print(f"Saved: {working_path}")
print(f"Saved: {exceptions_path}")

print("\nExceptions by source and reason:")
display(
    exceptions.groupby(
        ["source_table", "exception_reason"]
    ).size().rename("rows")
)

print("\nWorking data — first 10 rows:")
display(working_data.head(10))

print("\nExceptions — first 10 rows:")
display(exceptions.head(10))

Working accounts: 9,913
Excluded source rows: 260
Saved: C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\working data.csv
Saved: C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\exceptions.csv

Exceptions by source and reason:


source_table       exception_reason                        
applications       duplicate account_id after normalisation    24
                   no exact match in performance_12m           50
experian_consumer  no exact match in applications              37
                   no exact match in performance_12m           50
performance_12m    invalid account_id format                   37
                   missing account_id                          25
                   no exact match in applications              37
Name: rows, dtype: int64


Working data — first 10 rows:


,account_id,customer_id,customer_name,address_line_1,postcode,date_of_birth,age_at_application,application_date,employment_status,employment_tenure_months,...,missed_payment_m04,missed_payment_m05,missed_payment_m06,missed_payment_m07,missed_payment_m08,missed_payment_m09,missed_payment_m10,missed_payment_m11,missed_payment_m12,DelphiScore
0,ACC-10000000,CUS-20000000,Aisha Foster,218 Station Lane,NG14 9VP,1981-06-19,42,2024-06-02,part_time,26,...,0,0,0,0,0,0,0,0,0,825
1,ACC-10000001,CUS-20000001,Oliver Clarke,9 Kingfisher Way,LS27 2AP,1993-10-12,29,2023-05-29,retired,100,...,0,0,0,0,0,0,0,0,0,871
2,ACC-10000002,CUS-20000002,Isla Davies,181 Oak Road,CF29 7JL,1992-08-16,30,2023-04-11,self_employed,50,...,0,0,0,0,0,0,0,0,0,881
3,ACC-10000003,CUS-20000003,Maya Roberts,191 Station Lane,B3 8NV,1990-06-04,33,2023-12-27,retired,24,...,0,0,0,0,0,0,0,0,0,800
4,ACC-10000004,CUS-20000004,Priya Khan,205 Kingfisher Way,RG28 6EP,1990-12-07,33,2024-04-08,employed,25,...,0,0,0,0,0,0,0,0,0,841
5,ACC-10000005,CUS-20000005,Leo Clarke,151 Station Lane,RG2 2NF,1983-09-18,40,2024-03-11,employed,145,...,0,0,0,0,0,0,0,0,0,906
6,ACC-10000006,CUS-20000006,Maya Morgan,5 Victoria Road,NR5 3VN,1974-08-31,49,2024-01-06,retired,43,...,0,0,0,0,0,0,0,0,0,886
7,ACC-10000007,CUS-20000007,George Shah,4 Station Lane,PL19 1YV,1956-09-27,67,2023-12-11,employed,51,...,0,0,0,0,0,0,0,0,0,857
8,ACC-10000008,CUS-20000008,Sofia Davies,134 Park Avenue,PL24 2DB,1991-01-12,33,2024-11-21,employed,55,...,0,0,0,0,0,0,0,0,0,715
9,ACC-10000009,CUS-20000009,Adam Evans,61 Kingfisher Way,EH9 1GG,1988-09-24,35,2023-10-10,self_employed,32,...,0,0,0,0,0,0,0,0,0,762



Exceptions — first 10 rows:


,source_table,exception_reason,source_row_number,account_id,customer_id,customer_name,address_line_1,postcode,date_of_birth,age_at_application,...,missed_payment_m04,missed_payment_m05,missed_payment_m06,missed_payment_m07,missed_payment_m08,missed_payment_m09,missed_payment_m10,missed_payment_m11,missed_payment_m12,DelphiScore
0,applications,no exact match in performance_12m,134,ACC-10000133,CUS-20000133,Priya Lewis,170 Kingfisher Way,B23 2EP,1986-09-22,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,applications,duplicate account_id after normalisation,152,ACC-10000151,CUS-20000151,Aisha Wilson,221 Station Lane,NE24 6SJ,2000-02-13,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,applications,no exact match in performance_12m,172,ACC-10000171,CUS-20000171,Oliver Bennett,56 Meadow Close,EH8 9UD,1978-11-03,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,applications,no exact match in performance_12m,247,ACC-10000246,CUS-20000246,Leo Roberts,21 Kingfisher Way,OX24 8PU,1975-10-15,47.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,applications,no exact match in performance_12m,469,ACC-10000468,CUS-20000468,Jack Khan,200 Station Lane,LS22 8FQ,1961-08-16,62.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,applications,duplicate account_id after normalisation,517,ACC-10000516,CUS-20000516,Sofia Wilson,140 Willow Drive,EH9 6SX,1977-11-02,45.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,applications,no exact match in performance_12m,542,ACC-10000541,CUS-20000541,Freya Bennett,148 Park Avenue,NR20 4RD,1978-08-06,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,applications,no exact match in performance_12m,586,ACC-10000586,CUS-20000586,Daniel Patel,185 Mill View,NG28 5VT,1947-11-25,75.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,applications,no exact match in performance_12m,954,ACC-10000956,CUS-20000956,Leo Wilson,46 Park Avenue,NG11 9FT,1978-05-09,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,applications,duplicate account_id after normalisation,985,ACC-10000987,CUS-20000987,Isla Taylor,208 Willow Drive,B5 7UU,1978-02-19,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
